# B3 — B2 + LibriSpeech KenLM Beam Search

**No training required.** B3 is purely inference-time.

Takes the B2 model (WavLM fully fine-tuned on TORGO) and replaces
greedy CTC decoding with beam search + LibriSpeech 4-gram KenLM.

**Why LibriSpeech KenLM and not TORGO LM:**
A TORGO-trained LM would memorise the 957 repeated prompts and inflate
WER improvements artificially. LibriSpeech LM provides genuine linguistic
context (word co-occurrence, phonotactics) without prompt memorisation.

**Pipeline:**
1. Load B2 model
2. Download LibriSpeech 4-gram KenLM
3. Run B2 forward pass → CTC logits
4. Apply pyctcdecode beam search
5. Compare greedy vs beam WER per severity class
6. Grid search to find optimal α (LM weight) and β (word insertion bonus)

!pip install pyctcdecode==0.5.0
!pip install kenlm
# Pin numpy to a compatible version after kenlm install
!pip install numpy==1.26.4

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HOME"]            = "/kaggle/temp/hf"
os.environ["HF_DATASETS_CACHE"]  = "/kaggle/temp/hf/datasets"
os.environ["HF_HUB_CACHE"]       = "/kaggle/temp/hf/hub"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/temp/hf/transformers"
os.environ["XDG_CACHE_HOME"]     = "/kaggle/temp/.cache"

for p in [
    os.environ["HF_HOME"], os.environ["HF_DATASETS_CACHE"],
    os.environ["HF_HUB_CACHE"], os.environ["TRANSFORMERS_CACHE"],
    os.environ["XDG_CACHE_HOME"],
]:
    os.makedirs(p, exist_ok=True)

print("Cache dirs ready.")

Cache dirs ready.


#!pip install pyctcdecode==0.5.0
!pip install https://github.com/kpu/kenlm/archive/master.zip
!pip -q install -U transformers datasets evaluate jiwer soundfile huggingface_hub

In [2]:
import numpy as np
import pandas as pd
import torch
import evaluate
import json
import os
import random
from itertools import groupby
from collections import Counter

from datasets import load_dataset
from transformers import WavLMForCTC, Wav2Vec2Processor
from pyctcdecode import build_ctcdecoder
from huggingface_hub import hf_hub_download

import transformers
print("transformers:", transformers.__version__)
print("torch:",        torch.__version__)
print("GPU:",          torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

transformers: 5.6.1
torch: 2.10.0+cu128
GPU: True
GPU: Tesla T4


## Step 1 — Config

In [3]:
# ── Paths ──
B2_PATH = "/kaggle/input/datasets/kavishk/b2-pretrain/kaggle/working/b2v2_wavlm_ctc_final"

# ── TORGO config ──
RANDOM_SEED      = 42
TEST_SPEAKERS    = {"F01", "F04", "FC01", "M05"}
MLP_VAL_FRAC     = 0.2   # same val split as P2
CONTROL_TARGET   = 1500
MAX_AUDIO_SAMPLES = 240_000
SEVERITY_MAPPING = {
    "F01": "severe",   "M01": "severe",   "M02": "severe",   "M04": "severe",
    "M05": "moderate", "F03": "moderate",
    "F04": "mild",     "M03": "mild",
    "FC01": "control", "FC02": "control", "FC03": "control",
    "MC01": "control", "MC02": "control", "MC03": "control", "MC04": "control",
}

BEAM_WIDTH = 100

print("Config ready.")
print(f"B2_PATH: {B2_PATH}")


Config ready.
B2_PATH: /kaggle/input/datasets/kavishk/b2-pretrain/kaggle/working/b2v2_wavlm_ctc_final


## Step 2 — Load TORGO test split

In [4]:
import random
import pandas as pd
from collections import Counter

print("Loading TORGO...")
cache = os.environ["HF_DATASETS_CACHE"]
ds    = load_dataset("abnerh/TORGO-database", cache_dir=cache)
df    = ds["train"].to_pandas()

df["audio_path"] = df["audio"].apply(lambda x: x["path"])
df["speaker"]    = df["audio_path"].apply(lambda p: str(p).split("_")[0])
df["Category"]   = df["speaker"].map(SEVERITY_MAPPING)

hf_full    = ds["train"].add_column("speaker",  df["speaker"].tolist())
hf_full    = hf_full.add_column("Category", df["Category"].tolist())

# ── Test split (held-out speakers) ──
test_mask  = df["speaker"].isin(TEST_SPEAKERS)
torgo_test = hf_full.select(df[test_mask].index.tolist())
torgo_test = torgo_test.filter(
    lambda x: len(x["audio"]["array"]) <= MAX_AUDIO_SAMPLES, num_proc=1
)

# ── Val split — same as P2: stratified 20% of train pool ──
train_pool_df = df[~test_mask].copy()
ctrl  = train_pool_df[train_pool_df["Category"] == "control"].sample(
    n=min(CONTROL_TARGET, len(train_pool_df[train_pool_df["Category"] == "control"])),
    random_state=RANDOM_SEED
)
other = train_pool_df[train_pool_df["Category"] != "control"]
train_pool_df = pd.concat([ctrl, other]).sample(frac=1, random_state=RANDOM_SEED)

random.seed(RANDOM_SEED)
val_idx = []
for sev in ["control", "mild", "moderate", "severe"]:
    sev_idx = train_pool_df[train_pool_df["Category"] == sev].index.tolist()
    random.shuffle(sev_idx)
    n_val = max(1, int(len(sev_idx) * MLP_VAL_FRAC))
    val_idx.extend(sev_idx[:n_val])

mlp_val = hf_full.select(val_idx)
mlp_val = mlp_val.filter(
    lambda x: len(x["audio"]["array"]) <= MAX_AUDIO_SAMPLES, num_proc=1
)

print(f"Test utterances: {len(torgo_test)}")
print(f"Val utterances:  {len(mlp_val)}")
print("Test severity:", dict(sorted(Counter(torgo_test["Category"]).items())))
print("Val severity: ", dict(sorted(Counter(mlp_val["Category"]).items())))
print("Test speakers:", sorted(set(torgo_test["speaker"])))


Loading TORGO...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004.parquet:   0%|          | 0.00/370M [00:00<?, ?B/s]

data/train-00001-of-00004.parquet:   0%|          | 0.00/397M [00:00<?, ?B/s]

data/train-00002-of-00004.parquet:   0%|          | 0.00/356M [00:00<?, ?B/s]

data/train-00003-of-00004.parquet:   0%|          | 0.00/441M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16552 [00:00<?, ? examples/s]

Filter (num_proc=1):   0%|          | 0/1798 [00:00<?, ? examples/s]

Filter (num_proc=1):   0%|          | 0/1115 [00:00<?, ? examples/s]

Test utterances: 1786
Val utterances:  1111
Test severity: {'control': 302, 'mild': 673, 'moderate': 575, 'severe': 236}
Val severity:  {'control': 300, 'mild': 162, 'moderate': 217, 'severe': 432}
Test speakers: ['F01', 'F04', 'FC01', 'M05']


## Step 3 — Load B2 model

In [5]:
processor = Wav2Vec2Processor.from_pretrained(B2_PATH)
model     = WavLMForCTC.from_pretrained(
    B2_PATH,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)
model.eval()

total  = sum(p.numel() for p in model.parameters())
print(f"B2 model loaded on {device}")
print(f"Parameters: {total:,}")
print(f"Vocab size: {processor.tokenizer.vocab_size}")

Loading weights:   0%|          | 0/250 [00:00<?, ?it/s]

B2 model loaded on cuda
Parameters: 94,406,544
Vocab size: 30


## Step 4 — Download LibriSpeech KenLM

In [6]:
import warnings, logging
logging.getLogger("pyctcdecode").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

print("Downloading LibriSpeech KenLM + unigrams...")
lm_dir = "/kaggle/temp/kenlm"
os.makedirs(lm_dir, exist_ok=True)

lm_path = hf_hub_download(
    repo_id="jonatasgrosman/wav2vec2-large-xlsr-53-english",
    filename="language_model/lm.binary",
    cache_dir=lm_dir,
)
unigrams_path = hf_hub_download(
    repo_id="jonatasgrosman/wav2vec2-large-xlsr-53-english",
    filename="language_model/unigrams.txt",
    cache_dir=lm_dir,
)
unigrams = open(unigrams_path).read().strip().split("\n")
print(f"KenLM:    {lm_path}")
print(f"Unigrams: {unigrams_path}  ({len(unigrams):,} words)")


language_model/lm.binary:   0%|          | 0.00/863M [00:00<?, ?B/s]

language_model/unigrams.txt:   0%|          | 0.00/3.51M [00:00<?, ?B/s]

KenLM:    /kaggle/temp/kenlm/models--jonatasgrosman--wav2vec2-large-xlsr-53-english/snapshots/569a6236e92bd5f7652a0420bfe9bb94c5664080/language_model/lm.binary
Unigrams: /kaggle/temp/kenlm/models--jonatasgrosman--wav2vec2-large-xlsr-53-english/snapshots/569a6236e92bd5f7652a0420bfe9bb94c5664080/language_model/unigrams.txt  (373,978 words)


## Step 5 — Build vocabulary and decoder

In [7]:
from pyctcdecode import build_ctcdecoder

vocab_dict = processor.tokenizer.get_vocab()
vocab_list = [None] * len(vocab_dict)
for token, idx in vocab_dict.items():
    vocab_list[idx] = token
blank_id = processor.tokenizer.pad_token_id
vocab_list[blank_id] = ""

print(f"Vocab size: {len(vocab_list)}  blank_id: {blank_id}")

# Decoder cache — avoid rebuilding same (alpha, beta) repeatedly
_decoder_cache = {}

def get_decoder(alpha, beta):
    key = (round(float(alpha), 3), round(float(beta), 3))
    if key not in _decoder_cache:
        _decoder_cache[key] = build_ctcdecoder(
            labels=vocab_list,
            kenlm_model_path=lm_path,
            unigrams=unigrams,
            alpha=key[0],
            beta=key[1],
        )
    return _decoder_cache[key]

print("Decoder helper ready (with unigrams).")


Vocab size: 32  blank_id: 29
Decoder helper ready (with unigrams).


## Step 6 — Run B2 forward pass, collect logits + greedy baseline

In [8]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def decode_ctc_greedy(pred_ids, tokenizer):
    blank_id = tokenizer.pad_token_id
    decoded  = []
    for seq in pred_ids:
        collapsed = [k for k, _ in groupby(seq)]
        filtered  = [t for t in collapsed if t != blank_id]
        decoded.append(
            tokenizer.decode(filtered, skip_special_tokens=True) if filtered else ""
        )
    return decoded


def to_log_probs(logits_np):
    probs = np.exp(logits_np) / np.exp(logits_np).sum(axis=-1, keepdims=True)
    return np.log(np.clip(probs, 1e-8, 1.0))


def collect_logits(dataset, desc=""):
    logits_list, labels_list, cats_list, greedy_list = [], [], [], []
    BATCH = 8
    for i in range(0, len(dataset), BATCH):
        end     = min(i + BATCH, len(dataset))
        samples = [dataset[j] for j in range(i, end)]
        arrays  = [s["audio"]["array"] for s in samples]
        sr      = samples[0]["audio"]["sampling_rate"]
        inputs  = processor(arrays, sampling_rate=sr, return_tensors="pt", padding=True)
        iv      = inputs.input_values.to(device)
        am      = inputs.attention_mask.to(device) if "attention_mask" in inputs else None
        with torch.no_grad():
            out    = model(input_values=iv, attention_mask=am)
            logits = out.logits
        logits_list.extend([logits[j].cpu().numpy() for j in range(logits.size(0))])
        pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()
        greedy_list.extend([p.strip() for p in decode_ctc_greedy(pred_ids, processor.tokenizer)])
        labels_list.extend([s["transcription"].lower().strip() for s in samples])
        cats_list.extend([s["Category"] for s in samples])
        if (i // BATCH + 1) % 20 == 0:
            print(f"  [{desc}] {end}/{len(dataset)}")
    return logits_list, labels_list, cats_list, greedy_list


print("Collecting logits for val set (α/β tuning)...")
val_logits, val_labels, val_cats, val_greedy = collect_logits(mlp_val, desc="val")
print(f"Val: {len(val_logits)} utterances")

print("\nCollecting logits for test set (final eval)...")
all_logits, all_labels, all_cats, greedy_preds = collect_logits(torgo_test, desc="test")
print(f"Test: {len(all_logits)} utterances")

# Greedy baseline on test
eval_greedy = [p if p else "⟨empty⟩" for p in greedy_preds]
greedy_wer  = wer_metric.compute(predictions=eval_greedy, references=all_labels)
greedy_cer  = cer_metric.compute(predictions=eval_greedy, references=all_labels)
results_df  = pd.DataFrame({"prediction": greedy_preds, "reference": all_labels, "Category": all_cats})

print("\n" + "=" * 55)
print("B2 Greedy CTC — baseline")
print("=" * 55)
print(f"Overall WER: {greedy_wer*100:.2f}%   CER: {greedy_cer*100:.2f}%")
print("\nPer-severity:")
for cat in ["control", "mild", "moderate", "severe"]:
    sub = results_df[results_df["Category"] == cat]
    sub = sub[sub["reference"].str.strip().str.len() > 0]
    if len(sub) == 0: continue
    pg  = [p if p else "⟨empty⟩" for p in sub["prediction"].tolist()]
    gw  = wer_metric.compute(predictions=pg, references=sub["reference"].tolist())
    gc  = cer_metric.compute(predictions=pg, references=sub["reference"].tolist())
    print(f"  {cat:10s}: WER={gw*100:.2f}%  CER={gc*100:.2f}%  (n={len(sub)})")


  [val] 160/1111
  [val] 320/1111
  [val] 480/1111
  [val] 640/1111
  [val] 800/1111
  [val] 960/1111
Val: 1111 utterances

  [test] 160/1786
  [test] 320/1786
  [test] 480/1786
  [test] 640/1786
  [test] 800/1786
  [test] 960/1786
  [test] 1120/1786
  [test] 1280/1786
  [test] 1440/1786
  [test] 1600/1786
  [test] 1760/1786
Test: 1786 utterances

B2 Greedy CTC — baseline
Overall WER: 50.53%   CER: 24.15%

Per-severity:
  control   : WER=27.21%  CER=8.28%  (n=302)
  mild      : WER=31.30%  CER=9.80%  (n=673)
  moderate  : WER=78.30%  CER=43.77%  (n=575)
  severe    : WER=73.76%  CER=43.91%  (n=236)


## Step 7 — Bayesian optimisation on val set

Tunes a single shared (α, β) using the same val split as P2.
Uses true severity labels (oracle) — val set is not the test set.
Unigrams are active so β is properly functional.

In [9]:
!pip install -q scikit-optimize
from skopt import gp_minimize
from skopt.space import Real

GRID_BEAM = 50


def eval_params_on_val(alpha, beta):
    """WER on full val set with given (alpha, beta)."""    
    decoder = get_decoder(alpha, beta)
    preds   = []
    refs    = []
    for logits_np, label in zip(val_logits, val_labels):
        if not label.strip():
            continue
        text = decoder.decode(to_log_probs(logits_np), beam_width=GRID_BEAM)
        preds.append(text.strip().lower() or "⟨empty⟩")
        refs.append(label)
    return wer_metric.compute(predictions=preds, references=refs)


print("Bayesian optimisation on val set (unigrams active)...")
call_count = [0]

def objective(params):
    alpha, beta = params
    wer = eval_params_on_val(alpha, beta)
    call_count[0] += 1
    print(f"  [{call_count[0]:2d}] alpha={alpha:.3f}  beta={beta:.3f}  "
          f"WER={wer*100:.2f}%", flush=True)
    return wer

result = gp_minimize(
    func=objective,
    dimensions=[Real(0.0, 3.0, name="alpha"), Real(0.0, 2.0, name="beta")],
    n_calls=40,
    n_initial_points=10,
    acq_func="EI",
    random_state=42,
    verbose=False,
)

best_alpha = round(float(result.x[0]), 3)
best_beta  = round(float(result.x[1]), 3)
best_val_wer = float(result.fun)
print(f"\nBest (val): alpha={best_alpha}  beta={best_beta}  WER={best_val_wer*100:.2f}%")


Bayesian optimisation on val set (unigrams active)...
  [ 1] alpha=2.390  beta=0.367  WER=34.39%
  [ 2] alpha=2.339  beta=1.194  WER=33.28%
  [ 3] alpha=1.337  beta=0.200  WER=23.50%
  [ 4] alpha=1.378  beta=0.667  WER=23.27%
  [ 5] alpha=0.429  beta=1.302  WER=14.62%
  [ 6] alpha=0.169  beta=1.444  WER=16.76%
  [ 7] alpha=2.816  beta=0.002  WER=37.43%
  [ 8] alpha=2.977  beta=1.235  WER=37.13%
  [ 9] alpha=1.835  beta=0.014  WER=29.82%
  [10] alpha=0.069  beta=1.050  WER=20.68%
  [11] alpha=0.636  beta=2.000  WER=14.70%
  [12] alpha=0.590  beta=0.000  WER=14.93%
  [13] alpha=0.520  beta=1.966  WER=14.78%
  [14] alpha=0.730  beta=1.988  WER=15.19%
  [15] alpha=0.494  beta=0.008  WER=14.81%
  [16] alpha=0.310  beta=1.966  WER=14.93%
  [17] alpha=0.943  beta=1.951  WER=16.53%
  [18] alpha=0.381  beta=0.029  WER=14.78%
  [19] alpha=0.843  beta=1.881  WER=15.96%
  [20] alpha=1.080  beta=1.967  WER=17.90%
  [21] alpha=0.774  beta=0.002  WER=16.41%
  [22] alpha=0.273  beta=0.033  WER=15.23%


In [10]:
# ── Final eval on test set with val-tuned params ──
print(f"Running B3 on test set (alpha={best_alpha}, beta={best_beta}, beam={BEAM_WIDTH})...")
final_decoder = get_decoder(best_alpha, best_beta)

tuned_preds = []
for i, logits_np in enumerate(all_logits):
    text = final_decoder.decode(to_log_probs(logits_np), beam_width=BEAM_WIDTH)
    tuned_preds.append(text.strip().lower())
    if (i + 1) % 200 == 0:
        print(f"  {i+1}/{len(all_logits)}")

eval_tuned = [p if p else "⟨empty⟩" for p in tuned_preds]
tuned_wer  = wer_metric.compute(predictions=eval_tuned, references=all_labels)
tuned_cer  = cer_metric.compute(predictions=eval_tuned, references=all_labels)
tuned_df   = pd.DataFrame({"prediction": tuned_preds, "reference": all_labels, "Category": all_cats})

print("\n" + "=" * 65)
print("B3 FINAL — KenLM Beam Search (Bayesian-tuned α/β, with unigrams)")
print("=" * 65)
print(f"alpha={best_alpha}  beta={best_beta}  beam={BEAM_WIDTH}")
print(f"Overall WER: {tuned_wer*100:.2f}%   CER: {tuned_cer*100:.2f}%")
print(f"vs Greedy:   {greedy_wer*100:.2f}%         {greedy_cer*100:.2f}%")
print(f"Improvement: {(greedy_wer-tuned_wer)*100:.2f}pp WER  "
      f"{(greedy_cer-tuned_cer)*100:.2f}pp CER")
print("\nPer-severity:")
print("-" * 80)
for cat in ["control", "mild", "moderate", "severe"]:
    sub_g = results_df[results_df["Category"] == cat]
    sub_t = tuned_df[tuned_df["Category"] == cat]
    sub_g = sub_g[sub_g["reference"].str.strip().str.len() > 0]
    sub_t = sub_t[sub_t["reference"].str.strip().str.len() > 0]
    if len(sub_t) == 0: continue
    pg = [p if p else "⟨empty⟩" for p in sub_g["prediction"].tolist()]
    pt = [p if p else "⟨empty⟩" for p in sub_t["prediction"].tolist()]
    refs = sub_t["reference"].tolist()
    gw = wer_metric.compute(predictions=pg, references=refs)
    tw = wer_metric.compute(predictions=pt, references=refs)
    gc = cer_metric.compute(predictions=pg, references=refs)
    tc = cer_metric.compute(predictions=pt, references=refs)
    print(f"{cat:10s}: WER {gw*100:.2f}% → {tw*100:.2f}%  Δ={(gw-tw)*100:+.2f}pp   |"
          f"   CER {gc*100:.2f}% → {tc*100:.2f}%  Δ={(gc-tc)*100:+.2f}pp   (n={len(refs)})")


Running B3 on test set (alpha=0.459, beta=1.989, beam=100)...
  200/1786
  400/1786
  600/1786
  800/1786
  1000/1786
  1200/1786
  1400/1786
  1600/1786

B3 FINAL — KenLM Beam Search (Bayesian-tuned α/β, with unigrams)
alpha=0.459  beta=1.989  beam=100
Overall WER: 32.36%   CER: 19.09%
vs Greedy:   50.53%         24.15%
Improvement: 18.18pp WER  5.06pp CER

Per-severity:
--------------------------------------------------------------------------------
control   : WER 27.21% → 11.62%  Δ=+15.59pp   |   CER 8.28% → 4.72%  Δ=+3.56pp   (n=302)
mild      : WER 31.30% → 12.37%  Δ=+18.93pp   |   CER 9.80% → 5.31%  Δ=+4.50pp   (n=673)
moderate  : WER 78.30% → 59.56%  Δ=+18.74pp   |   CER 43.77% → 37.34%  Δ=+6.43pp   (n=575)
severe    : WER 73.76% → 56.03%  Δ=+17.73pp   |   CER 43.91% → 38.38%  Δ=+5.53pp   (n=236)


## Step 8 — Save results

In [11]:
# Save
tuned_df.to_csv("/kaggle/working/b3_torgo_test_results.csv", index=False)
summary = {
    "model":              "B3_KenLM_BeamSearch",
    "base_model":         "B2 (WavLM full fine-tune on TORGO)",
    "lm":                 "LibriSpeech 4-gram KenLM + unigrams",
    "alpha_beta_tuned_on": "mlp_val (same as P2, true labels, Bayesian opt)",
    "best_alpha":         best_alpha,
    "best_beta":          best_beta,
    "beam_width":         BEAM_WIDTH,
    "greedy_wer":         round(greedy_wer, 4),
    "greedy_cer":         round(greedy_cer, 4),
    "b3_wer":             round(tuned_wer, 4),
    "b3_cer":             round(tuned_cer, 4),
    "wer_improvement_pp": round((greedy_wer - tuned_wer) * 100, 2),
    "cer_improvement_pp": round((greedy_cer - tuned_cer) * 100, 2),
}
with open("/kaggle/working/b3_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Results saved.")
print(json.dumps(summary, indent=2))


Results saved.
{
  "model": "B3_KenLM_BeamSearch",
  "base_model": "B2 (WavLM full fine-tune on TORGO)",
  "lm": "LibriSpeech 4-gram KenLM + unigrams",
  "alpha_beta_tuned_on": "mlp_val (same as P2, true labels, Bayesian opt)",
  "best_alpha": 0.459,
  "best_beta": 1.989,
  "beam_width": 100,
  "greedy_wer": 0.5053,
  "greedy_cer": 0.2415,
  "b3_wer": 0.3236,
  "b3_cer": 0.1909,
  "wer_improvement_pp": 18.18,
  "cer_improvement_pp": 5.06
}
